# Preprocessing sandbox

Shows scenes before and after the model's preprocessing (`preprocess.get_common_bands`, then
`normalize_rasters` and `resize_rasters`), band by band with histograms. The first section draws random
scenes; the second shows hand-picked difficult ones. Set `USE_SAMPLE` to run on the committed sample.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from glacierview import config, preprocess, rasters

# True: the committed sample (Trient's full series plus one scene each for 100 more glaciers).
# False: the full local training landing zone.
USE_SAMPLE = False

BANDS = ["blue", "green", "red", "nir"]
if USE_SAMPLE:
    landsat = config.landsat_dir(config.INFERENCE_DATA_LABEL, root=config.SAMPLE_INFERENCE_ROOT)
    scenes = pd.read_csv(config.SAMPLE_INFERENCE_ROOT / "glaciers.csv")
else:
    landsat = config.landsat_dir(config.TRAINING_DATA_LABEL)
    metadata_dir = config.processed_metadata_dir(config.TRAINING_DATA_LABEL)
    scenes = pd.read_csv(metadata_dir / "filtered_training_data.csv")


def show(raster, title, xlim):
    fig, axs = plt.subplots(2, len(BANDS), figsize=(15, 6))
    fig.suptitle(title)
    for i, band in enumerate(BANDS):
        axs[0, i].imshow(raster[:, :, i], cmap="gray")
        axs[0, i].set_title(band)
        axs[1, i].hist(raster[:, :, i].ravel(), bins=25)
        axs[1, i].set_xlim(xlim)
    plt.show()


def compare(glims_id, file_name):
    scene = rasters.get_rasters(landsat / glims_id, file_name=file_name)
    raw = preprocess.get_common_bands(scene, BANDS)
    processed = preprocess.resize_rasters(preprocess.normalize_rasters(raw), config.MODEL_INPUT_DIM)
    show(raw[file_name], f"{file_name}: raw", (0, 70_000))
    show(processed[file_name], f"{file_name}: preprocessed", (0, 1))

## Random scenes

In [ ]:
for scene in scenes.sample(10).itertuples():
    compare(scene.glims_id, scene.file_name)

## Hand-picked difficult scenes

In [ ]:
hard = [
    ("G076336E33701N", "G076336E33701N_2007-10-10_L7_C02_T1_L2_SR.tif"),
    ("G277921E80969N", "G277921E80969N_2015-08-30_L8_C02_T1_L2_SR.tif"),
    ("G071284E36211N", "G071284E36211N_2006-09-01_L7_C02_T1_L2_SR.tif"),
    ("G082962E30869N", "G082962E30869N_2009-11-05_L5_C02_T1_L2_SR.tif"),
]
if USE_SAMPLE:  # those aren't in the sample; take one glacier per region instead
    hard = list(scenes.groupby("region").head(1)[["glims_id", "file_name"]].itertuples(index=False))
for glims_id, file_name in hard:
    compare(glims_id, file_name)